In [1]:
from pyspark.sql import SparkSession
from delta import * # Necessário para consultar tabelas Delta
from pyspark.sql import functions as F

# 1. CONFIGURAÇÃO DA SPARKSESSION
# Usamos a configuração do seu script anterior para garantir consistência.
warehouse_location = "hdfs://hdfs-nn:9000/demo/silver/delta_warehouse"
db_name = "demo"
table_name = "titles_delta"

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Titles Delta Querying and Updating") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession com suporte Hive e Delta Lake iniciada.")
spark.sql(f"USE {db_name}") # Definir a base de dados padrão

# 2. Investigar os dados da Tabela Delta: age_certification (análogo ao status)
print(f"\n--- 2. Valores distintos em 'age_certification' na tabela {table_name}: ---")
spark.sql(f"SELECT DISTINCT age_certification FROM {table_name}").show()

# 3. Consultar a Tabela Delta (COM O FILTRO CORRETO)
# Usando a coluna 'age_certification' com o valor 'R' como exemplo de filtro
print(f"\n--- 3. A consultar a tabela Delta (com filtro 'age_certification = R'): ---")
spark.sql(
    f"""
    SELECT title, type, release_year, age_certification, imdb_score
    FROM {table_name}
    WHERE age_certification = 'R'
    LIMIT 10
    """
).show()

# 4. Exemplo de SQL Avançado (SÓ FUNCIONA EM DELTA)
print(f"\n--- 4. A executar um UPDATE na tabela Delta ({table_name})... ---")

# Vamos corrigir ou atualizar um dado.
# Exemplo: Atualizar a pontuação IMDb do filme 'Taxi Driver' para 9.5
# Esta operação irá criar uma nova versão no Histórico do Delta
spark.sql(
    f"""
    UPDATE {table_name}
    SET imdb_score = 9.5
    WHERE title = 'Taxi Driver'
    """
)

print("Dados atualizados. A verificar o histórico novamente:")
spark.sql(f"DESCRIBE HISTORY {table_name}").show(truncate=False)

# 5. Consultar a Tabela Delta para verificar a atualização (Versão 1)
print("\n--- 5. Consulta para verificar a atualização: ---")
spark.sql(f"""
    SELECT title, imdb_score
    FROM {table_name}
    WHERE title = 'Taxi Driver'
""").show()


SparkSession com suporte Hive e Delta Lake iniciada.

--- 2. Valores distintos em 'age_certification' na tabela titles_delta: ---
+-----------------+
|age_certification|
+-----------------+
|             TV-Y|
|             null|
|               PG|
|            TV-MA|
|            TV-PG|
|            NC-17|
|                R|
|                G|
|            TV-14|
|             TV-G|
|            TV-Y7|
|            PG-13|
+-----------------+


--- 3. A consultar a tabela Delta (com filtro 'age_certification = R'): ---
+-----------------+-----+------------+-----------------+----------+
|            title| type|release_year|age_certification|imdb_score|
+-----------------+-----+------------+-----------------+----------+
|    The Gentlemen|MOVIE|        2019|                R|       7.8|
|     The Irishman|MOVIE|        2019|                R|       7.8|
|       Synchronic|MOVIE|        2019|                R|       6.2|
|   Marriage Story|MOVIE|        2019|                R|       7

In [7]:
from pyspark.sql import SparkSession
from delta import *
from pyspark.sql import functions as F # Importar F para futuras manipulações (agora necessário para ler Delta)

# 1. Iniciar a SparkSession (necessário para falar com o Hive e Delta)
# Usamos a configuração do seu ambiente de trabalho.
warehouse_location = 'hdfs://hdfs-nn:9000/demo/silver/delta_warehouse' # Localização do warehouse
db_name = "demo"
table_name = "titles_parquet"
delta_table_name = "titles_delta" # Nome da sua tabela Delta existente
delta_read_path = "hdfs://hdfs-nn:9000/demo/silver/titles_delta" # Localização dos ficheiros Delta

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Fix Titles Parquet Table Definition") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada. A corrigir a definição da tabela Parquet...")
spark.sql(f"USE {db_name}")

# 2. Apagar a definição errada da tabela (se existir)
spark.sql(f"DROP TABLE IF EXISTS {db_name}.{table_name}")

# 3. Registar a tabela corretamente
# Esta localização deve apontar para a pasta HDFS ONDE OS SEUS FICHEIROS PARQUET ESTÃO.
location_dos_ficheiros = "hdfs://hdfs-nn:9000/demo/silver/titles_parquet_data"

# --- INÍCIO DA DEFINIÇÃO DA TABELA ---
# Criamos a tabela Parquet particionada por 'release_year'
spark.sql(f"""
    CREATE EXTERNAL TABLE IF NOT EXISTS {db_name}.{table_name}
    (
        id STRING,
        title STRING,
        type STRING,
        description STRING,
        age_certification STRING,
        runtime INT,
        genres STRING,
        production_countries STRING,
        seasons INT,
        imdb_id STRING,
        imdb_score DOUBLE,
        imdb_votes DOUBLE,
        tmdb_popularity DOUBLE,
        tmdb_score DOUBLE
    )
    PARTITIONED BY (release_year INT)
    STORED AS PARQUET
    LOCATION '{location_dos_ficheiros}'
""")
# --- FIM DA DEFINIÇÃO DA TABELA ---

# 3.5. (NOVA ETAPA) TRANSFERIR DADOS DA DELTA PARA PARQUET
print(f"\n--- A TRANSFERIR DADOS: {delta_table_name} (Delta) -> {table_name} (Parquet) ---")

# Lemos a sua tabela Delta existente
delta_df = spark.read.format("delta").load(delta_read_path)

# Escrevemos os dados Delta como Parquet particionado no local definido
delta_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .partitionBy("release_year") \
    .option("path", location_dos_ficheiros) \
    .save()
    
print("Dados Delta escritos com sucesso como Parquet particionado.")

# 4. Dizer ao Hive para descobrir as partições
# O 'release_year' é usado aqui como coluna de partição.
spark.catalog.recoverPartitions(f"{db_name}.{table_name}")

print(f"\nDefinição da tabela Parquet '{db_name}.{table_name}' corrigida e partições recuperadas!")

# 5. Tente a consulta novamente (agora deve mostrar os dados do Parquet)
print("\n--- A CONSULTAR A TABELA PARQUET CORRIGIDA (DEVE FUNCIONAR AGORA) ---")
spark.sql(f"""
    SELECT title, type, release_year
    FROM {db_name}.{table_name}
    WHERE release_year = 2020
    LIMIT 10
""").show()

# 6. Parar a sessão (Mantenha comentada se estiver a usar um notebook)
# spark.stop()

SparkSession iniciada. A corrigir a definição da tabela Parquet...

--- A TRANSFERIR DADOS: titles_delta (Delta) -> titles_parquet (Parquet) ---
Dados Delta escritos com sucesso como Parquet particionado.

Definição da tabela Parquet 'demo.titles_parquet' corrigida e partições recuperadas!

--- A CONSULTAR A TABELA PARQUET CORRIGIDA (DEVE FUNCIONAR AGORA) ---
+--------------------+-----+------------+
|               title| type|release_year|
+--------------------+-----+------------+
|The Secret of Ski...| SHOW|        2020|
|Jurassic World: C...| SHOW|        2020|
|          Bridgerton| SHOW|        2020|
|           Control Z| SHOW|        2020|
|  The Queen's Gambit| SHOW|        2020|
|      The Last Dance| SHOW|        2020|
|         Locke & Key| SHOW|        2020|
|         Outer Banks| SHOW|        2020|
|      Emily in Paris| SHOW|        2020|
|          The Rental|MOVIE|        2020|
+--------------------+-----+------------+



In [13]:
from pyspark.sql import SparkSession
from delta import *

# 1. Configuração da SparkSession (necessária para todas as operações)
# Usamos o seu warehouse_location específico do Silver Layer
warehouse_location = 'hdfs://hdfs-nn:9000/demo/silver/delta_warehouse'
db_name = "demo"
table_name = "titles_delta" # A sua tabela principal

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Titles Delta SQL Analysis") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada. A executar consultas...")

# 2. Defina a base de dados
spark.sql(f"USE {db_name}")

# 3. COLOQUE AQUI AS SUAS CONSULTAS SQL
# Exemplo 1: Contagem de títulos por Classificação Etária (Age Certification)
print(f"\n--- Contagem de títulos por Classificação Etária na tabela {table_name}: ---")
spark.sql(
    f"""
    SELECT age_certification, COUNT(*) AS total_titulos
    FROM {table_name}
    GROUP BY age_certification
    ORDER BY total_titulos DESC
    """
).show()

# Exemplo 2: Pontuação Média IMDb por Tipo (Filme vs. Show)
print("\n--- Pontuação Média IMDb por Tipo de Conteúdo (MOVIE vs. SHOW): ---")
spark.sql(
    f"""
    SELECT type, ROUND(AVG(imdb_score), 2) AS media_imdb
    FROM {table_name}
    GROUP BY type
    """
).show()

SparkSession iniciada. A executar consultas...

--- Contagem de títulos por Classificação Etária na tabela titles_delta: ---
+-----------------+-------------+
|age_certification|total_titulos|
+-----------------+-------------+
|             null|         2619|
|            TV-MA|          883|
|                R|          556|
|            TV-14|          474|
|            PG-13|          451|
|               PG|          233|
|            TV-PG|          188|
|                G|          124|
|            TV-Y7|          120|
|             TV-Y|          107|
|             TV-G|           79|
|            NC-17|           16|
+-----------------+-------------+


--- Pontuação Média IMDb por Tipo de Conteúdo (MOVIE vs. SHOW): ---
+-----+----------+
| type|media_imdb|
+-----+----------+
| SHOW|      6.98|
|MOVIE|      6.25|
+-----+----------+



In [15]:
from pyspark.sql import SparkSession
from delta import *

# 1. Configuração da SparkSession (necessária para falar com o Hive e Delta)
# Usamos a localização específica do seu Silver Layer
warehouse_location = 'hdfs://hdfs-nn:9000/demo/silver/delta_warehouse'
db_name = "demo"
table_name = "titles_delta" # A sua tabela principal

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Describe Titles Delta Table") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada.")

# 2. O Comando para ver as colunas
print(f"\n--- A descrever a estrutura da tabela '{db_name}.{table_name}': ---")

# O comando DESCRIBE mostra o nome da coluna, o tipo de dados e o comentário
spark.sql(f"DESCRIBE {db_name}.{table_name}").show(truncate=False)

SparkSession iniciada.

--- A descrever a estrutura da tabela 'demo.titles_delta': ---
+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|id                     |string   |null   |
|title                  |string   |null   |
|type                   |string   |null   |
|description            |string   |null   |
|release_year           |int      |null   |
|age_certification      |string   |null   |
|runtime                |int      |null   |
|genres                 |string   |null   |
|production_countries   |string   |null   |
|seasons                |int      |null   |
|imdb_id                |string   |null   |
|imdb_score             |double   |null   |
|imdb_votes             |double   |null   |
|tmdb_popularity        |double   |null   |
|tmdb_score             |double   |null   |
|# Partition Information|         |       |
|# col_name             |data_type|comment|
|release_year           |int     

In [16]:
from pyspark.sql import SparkSession
from delta import *
from pyspark.sql.functions import col

# 1. Configuração da SparkSession (necessária para todas as operações)
# Usamos o seu warehouse_location específico do Silver Layer
warehouse_location = 'hdfs://hdfs-nn:9000/demo/silver/delta_warehouse'
db_name = "demo"
table_name = "titles_delta" # A sua tabela principal

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Titles Delta SQL Analysis - Release Year Trends") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada. A executar consulta SQL...")
spark.sql(f"USE {db_name}")

# 3. Consulta SQL: Análise de Tendências por Ano de Lançamento
# Esta consulta agrupa os filmes por 'release_year' para calcular as médias de duração e pontuação.
print(f"\n--- Análise de Duração e Qualidade Média por Ano de Lançamento na tabela {table_name}: ---")
spark.sql(
    f"""
    SELECT
        release_year,
        
        -- Calcular a duração média dos filmes para o ano
        ROUND(AVG(runtime), 2) AS media_duracao,
        
        -- Contar o número de filmes lançados nesse ano
        COUNT(*) AS total_filmes,
        
        -- Calcular a média da pontuação IMDb para o ano
        ROUND(AVG(imdb_score), 2) AS media_imdb
        
    FROM {table_name}
    -- Filtramos apenas por MOVIEs e garantimos que os dados não são nulos para 'runtime' e 'imdb_score'
    WHERE type = 'MOVIE' AND runtime IS NOT NULL AND imdb_score IS NOT NULL
    GROUP BY release_year
    ORDER BY release_year DESC -- Ordena do mais recente para o mais antigo
    """
).show()

SparkSession iniciada. A executar consulta SQL...

--- Análise de Duração e Qualidade Média por Ano de Lançamento na tabela titles_delta: ---
+------------+-------------+------------+----------+
|release_year|media_duracao|total_filmes|media_imdb|
+------------+-------------+------------+----------+
|        2022|       100.26|         180|      5.95|
|        2021|        98.74|         429|      6.01|
|        2020|        95.49|         424|      6.01|
|        2019|        97.45|         465|      6.22|
|        2018|        98.12|         446|      6.21|
|        2017|        96.62|         367|      6.21|
|        2016|        96.44|         213|      6.22|
|        2015|       103.22|         124|      6.29|
|        2014|       104.35|         103|      6.63|
|        2013|       105.63|          92|      6.46|
|        2012|       111.56|          73|      6.51|
|        2011|       108.49|          55|      6.57|
|        2010|       109.02|          45|      6.62|
|        2